## Let's get started Weight & Biases
This sample code is designed to help you learn Weights & Biases (WandB) step by step.
It includes practical preprocessing for LightGBM, model training, and submission generation.

While this code is already useful as it is, you can turn it into a project capable of experiment tracking by adding code for WandB integration.
This notebook represents the “before” version. For the “after” version, please refer to the following link:

[https://www.kaggle.com/code/yutodennou/practice-wandb-lightgbm-whole-codes](https://www.kaggle.com/code/yutodennou/practice-wandb-lightgbm-whole-codes)

------

このサンプルコードは、Weights & Biases (WandB) を段階的に学ぶためのものです。
LightGBM を用いた前処理、モデル学習、そして提出ファイル（submission）の作成までを含んでいます。

コードはそのままでも十分に有用ですが、WandBと連携するコードを追加することで、実験結果を管理できるプロジェクトへと発展させることができます。
このNotebookは練習用なので編集後のNotebookはこちらを参考にしてください。  
[https://www.kaggle.com/code/yutodennou/practice-wandb-lightgbm-whole-codes](https://www.kaggle.com/code/yutodennou/practice-wandb-lightgbm-whole-codes)

In [ ]:
# =========================================================
# Store Sales - Time Series Forecasting: LightGBM Baseline
# EN: End-to-end baseline: load data → feature engineering → time-series CV → train → predict → submission.csv
# JA: 端から端までのベースライン：データ読込 → 特徴量作成 → 時系列CV → 学習 → 予測 → submission.csv 出力
# Environment: Kaggle Notebook / Python, LightGBM
# =========================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error

In [ ]:
# -----------------------------
# Utilities / ユーティリティ
# -----------------------------
def reduce_mem(df):
    """
    EN: Downcast numeric dtypes to reduce memory footprint.
    JA: 数値型をダウンキャストしてメモリ消費を削減します。
    """
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
        elif col_type == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
    return df

def rmsle(y_true, y_pred):
    """
    EN: Compute RMSLE with clipping at 0 to avoid negatives.
    JA: 負値を0でクリップした上でRMSLEを計算します。
    """
    return np.sqrt(mean_squared_log_error(np.maximum(y_true, 0), np.maximum(y_pred, 0)))

def add_date_features(df, date_col='date'):
    """
    EN: Add basic calendar features from a datetime column.
    JA: 日付列から基本的なカレンダー特徴量を追加します。
    """
    df['doy']   = df[date_col].dt.dayofyear
    df['woy']   = df[date_col].dt.isocalendar().week.astype(int)
    df['dom']   = df[date_col].dt.day
    df['dow']   = df[date_col].dt.dayofweek     # 0=Mon,...,6=Sun
    df['month'] = df[date_col].dt.month
    df['year']  = df[date_col].dt.year
    df['is_weekend'] = (df['dow'] >= 5).astype('int8')
    return df

def make_lag_features(df, key_cols, target_col='sales', lags=(7,14,28), roll_windows=(7,14,28)):
    """
    EN: Create lags and rolling stats per (store_nbr, family) group.
    JA: （store_nbr, family）単位でラグ・移動統計量を作成します。
    """
    df = df.sort_values(['store_nbr', 'family', 'date'])
    g = df.groupby(key_cols, observed=True)

    for L in lags:
        df[f'lag_{L}'] = g[target_col].shift(L)

    for w in roll_windows:
        df[f'roll_mean_{w}']   = g[target_col].shift(1).rolling(w).mean()
        df[f'roll_median_{w}'] = g[target_col].shift(1).rolling(w).median()
        df[f'roll_max_{w}']    = g[target_col].shift(1).rolling(w).max()
        df[f'roll_min_{w}']    = g[target_col].shift(1).rolling(w).min()
        df[f'roll_std_{w}']    = g[target_col].shift(1).rolling(w).std()

    return df

In [ ]:
# -----------------------------
# 1) Load data / データ読み込み
# -----------------------------
INPUT_DIR = "/kaggle/input/store-sales-time-series-forecasting"

train = pd.read_csv(f"{INPUT_DIR}/train.csv", parse_dates=['date'])
test  = pd.read_csv(f"{INPUT_DIR}/test.csv",  parse_dates=['date'])
stores = pd.read_csv(f"{INPUT_DIR}/stores.csv")
oil = pd.read_csv(f"{INPUT_DIR}/oil.csv", parse_dates=['date'])
holidays = pd.read_csv(f"{INPUT_DIR}/holidays_events.csv", parse_dates=['date'])
transactions = pd.read_csv(f"{INPUT_DIR}/transactions.csv", parse_dates=['date'])

# Lighten memory / メモリ軽量化
train = reduce_mem(train)
test  = reduce_mem(test)
stores = reduce_mem(stores)
oil = reduce_mem(oil)
holidays = reduce_mem(holidays)
transactions = reduce_mem(transactions)

# Notes (spec) / 仕様メモ:
# EN: train: [date, store_nbr, family, sales, onpromotion], test: [id, date, store_nbr, family, onpromotion], submit: [id, sales]
# JA: train/test の列は上記の通り。提出は [id, sales]。

In [ ]:
# -----------------------------
# 2) Join masters & preprocess / マスタ結合・前処理
# -----------------------------
# EN: Categorical casting for store metadata.
# JA: 店舗メタのカテゴリ化。
for c in ['city', 'state', 'type']:
    if c in stores.columns:
        stores[c] = stores[c].astype('category')

# EN: Holidays → simple "is_holiday" flag (max per date).
# JA: 祝日データを日付ごとのフラグに簡略化。
holidays['is_holiday'] = 1
holidays_simple = holidays[['date','is_holiday']].groupby('date', as_index=False).max()

# EN: Oil price: interpolate missing & add moving averages.
# JA: 原油価格を補間し、移動平均を追加。
oil = oil.rename(columns={'dcoilwtico':'oil_price'})
oil['oil_price'] = oil['oil_price'].interpolate(limit_direction='both')
oil['oil_ma7']   = oil['oil_price'].rolling(7,  min_periods=1).mean()
oil['oil_ma30']  = oil['oil_price'].rolling(30, min_periods=1).mean()

# EN: Transactions: add per-store moving averages.
# JA: 店舗別の取引数に移動平均を付与。
transactions = transactions.sort_values(['store_nbr','date'])
transactions['trans_ma7']  = transactions.groupby('store_nbr')['transactions'].transform(lambda s: s.rolling(7,  min_periods=1).mean())
transactions['trans_ma30'] = transactions.groupby('store_nbr')['transactions'].transform(lambda s: s.rolling(30, min_periods=1).mean())

# EN: Concat train & test to build features at once (keep sales NaN for test).
# JA: 学習・テストを結合して一括で特徴量作成（テストの sales は欠損のまま）。
train['is_train'] = 1
test['is_train']  = 0
data = pd.concat([train, test], axis=0, ignore_index=True)

# EN: Join stores / JA: 店舗データ結合
data = data.merge(stores, on='store_nbr', how='left')

# EN: Join holidays / JA: 祝日フラグ結合
data = data.merge(holidays_simple, on='date', how='left')
data['is_holiday'] = data['is_holiday'].fillna(0).astype('int8')

# EN: Join oil features / JA: 原油価格特徴の結合
data = data.merge(oil[['date','oil_price','oil_ma7','oil_ma30']], on='date', how='left')
data[['oil_price','oil_ma7','oil_ma30']] = data[['oil_price','oil_ma7','oil_ma30']].fillna(method='ffill').fillna(method='bfill')

# EN: Join transactions (per store, not per family) / JA: 取引数（店舗単位）を結合
data = data.merge(transactions[['store_nbr','date','transactions','trans_ma7','trans_ma30']], on=['store_nbr','date'], how='left')
for c in ['transactions','trans_ma7','trans_ma30']:
    data[c] = data[c].fillna(0)

# EN: Label-encode selected categoricals.
# JA: 必要なカテゴリ列をラベルエンコード。
for c in ['family','city','state','type']:
    if c in data.columns:
        le = LabelEncoder()
        data[c] = le.fit_transform(data[c].astype(str))

# EN: Calendar features / JA: カレンダー特徴
data = add_date_features(data, 'date')

# EN: Fill onpromotion if any missing / JA: onpromotion の欠損埋め
if 'onpromotion' in data.columns:
    data['onpromotion'] = data['onpromotion'].fillna(0).astype('float32')

In [ ]:
# -----------------------------
# 3) Lags & rolling statistics / ラグ・移動統計
# -----------------------------
# EN: Keep sales NaN on test to avoid leakage; compute group-based shifts.
# JA: リーク回避のため test の sales は欠損のまま、グループごとの shift を計算。
data = data.sort_values(['store_nbr','family','date'])
data = make_lag_features(
    data,
    key_cols=['store_nbr','family'],
    target_col='sales',
    lags=(7,14,28),
    roll_windows=(7,14,28)
)

# EN: Create lags for onpromotion/transactions as exogenous signals.
# JA: 外生情報として onpromotion / transactions にもラグを作成。
for L in [7,14]:
    data[f'onpromo_lag_{L}'] = data.groupby(['store_nbr','family'])['onpromotion'].shift(L)
    data[f'trans_lag_{L}']   = data.groupby(['store_nbr'])['transactions'].shift(L)

# EN: Fill NaNs for newly created lag/rolling columns.
# JA: 新規作成したラグ/移動統計列の欠損を0埋め。
lag_cols = [c for c in data.columns if 'lag_' in c or 'roll_' in c]
fill_zero_cols = lag_cols + [c for c in data.columns if c.startswith('onpromo_lag_') or c.startswith('trans_lag_')]
for c in fill_zero_cols:
    data[c] = data[c].fillna(0)

In [ ]:
# -----------------------------
# 4) Split train/valid/test / 学習・検証・テスト分割
# -----------------------------
train_df = data[data['is_train']==1].copy()
test_df  = data[data['is_train']==0].copy()

# EN: Use last 28 days of train as validation (simple, fast).
# JA: 学習データの最後28日を検証に使用（簡易・高速）。
last_date = train_df['date'].max()
val_from = last_date - pd.Timedelta(days=27)
train_part = train_df[train_df['date'] < val_from]
valid_part = train_df[train_df['date'] >= val_from]

# EN: Feature columns / JA: 特徴量カラム
drop_cols = ['id', 'date', 'sales', 'is_train']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_part[feature_cols]
y_train = train_part['sales']
X_valid = valid_part[feature_cols]
y_valid = valid_part['sales']

In [ ]:
# -----------------------------
# 5) Train LightGBM / LightGBM学習
# -----------------------------
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'min_data_in_leaf': 50,
    'verbose': -1,
    'seed': 42,
}

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

evals_result = {}

callbacks = [
    lgb.early_stopping(stopping_rounds=200, verbose=True),
    lgb.log_evaluation(period=200),
    lgb.record_evaluation(evals_result) 
]

model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train','valid'],
    callbacks=callbacks
)

# EN: Report RMSLE for reference (competition uses RMSE, but RMSLE helps sanity-check).
# JA: 参考としてRMSLEを表示（本番指標はRMSEだが検算に有用）。
valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)
val_rmsle = rmsle(y_valid.values, np.maximum(valid_pred, 0))
print(f"[Validation] RMSLE: {val_rmsle:.6f}")


In [ ]:
train_rmse = evals_result['train']['rmse']
valid_rmse = evals_result['valid']['rmse']

plt.figure(figsize=(8, 5))
plt.plot(train_rmse, label='Train RMSE')
plt.plot(valid_rmse, label='Valid RMSE')
plt.xlabel('Iteration (boosting round)')   # JA: 反復（ブースティング回数）
plt.ylabel('RMSE')                         # JA: RMSE（平方根平均二乗誤差）
plt.title('LightGBM Learning Curves')      # JA: LightGBM 学習曲線
plt.legend()
plt.grid(True)
plt.tight_layout()

In [ ]:
# -----------------------------
# 6) Predict & submit / 予測と提出ファイル作成
# -----------------------------
X_test = test_df[feature_cols]
test_pred = model.predict(X_test, num_iteration=model.best_iteration)
test_pred = np.maximum(test_pred, 0)  # EN: clip negatives / JA: 負値クリップ

sub = pd.DataFrame({
    'id': test_df['id'].astype(int),
    'sales': test_pred
}).sort_values('id')

sub.to_csv('submission.csv', index=False)
print("Saved: submission.csv")

### Once you’ve made it this far, let’s move on to a project that uses Weights & Biases (WandB).
#### 👉 [Here, you can learn WandB](https://www.kaggle.com/code/yutodennou/practice-wandb-lightgbm-whole-codes)